# BOT RAG + Agente + Mistral


Bloque 1 — Instalaciones

In [1]:
!pip install -q langchain-core==0.3.63
!pip install -q langchain==0.3.25
!pip install -q langchain-community==0.3.24
!pip install -q langchain-mistralai==0.2.10
!pip install -q faiss-cpu==1.11.0
!pip install -q langgraph==0.4.7
!pip install -q mistralai

print("✅ Todas las dependencias instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 81.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.63 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.63 which is incompatible.
langchain 1.2.15 requires langchain-core<2.0.0,>=1.2.10, but you have langchain-core 0.3.63 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 25.6 MB/s eta 0:00:00
ERROR: pip

Bloque 2 — Reiniciar el kernel

In [ ]:
import os
os.kill(os.getpid(), 9)

Bloque 3 — Imports

In [2]:
import subprocess
paquetes = ['langchain', 'langchain-core', 'langchain-community',
            'langchain-mistralai', 'langgraph', 'mistralai']

for p in paquetes:
    result = subprocess.run(['pip', 'show', p], capture_output=True, text=True)
    for line in result.stdout.split('\n'):
        if line.startswith('Version'):
            print(f"{p}: {line}")

langchain: Version: 0.3.25
langchain-core: Version: 1.4.0
langchain-community: Version: 0.3.24
langchain-mistralai: Version: 0.2.10
langgraph: Version: 0.4.7
mistralai: Version: 2.4.5


In [3]:
!pip install -q --force-reinstall \
    langchain==0.3.25 \
    langchain-core==0.3.63 \
    langchain-community==0.3.24 \
    langchain-mistralai==0.2.10 \
    langgraph==0.2.73 \
    mistralai==1.2.5

print("✅ Versiones compatibles instaladas")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.5/151.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import os
import time
import pandas as pd
from langchain_core.documents import Document
from langchain_mistralai import MistralAIEmbeddings, ChatMistralAI
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage, AIMessage
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent
import ipywidgets as widgets
from IPython.display import display, HTML

print("✅ Imports correctos")

✅ Imports correctos


Bloque 4 — API Key y Dataset

In [4]:
from google.colab import userdata, files

# API Key desde Secrets de Colab
os.environ["MISTRAL_API_KEY"] = userdata.get('bot-ventas-rag')
print("✅ API Key cargada desde Secrets")

# Subir dataset
uploaded = files.upload()  # Subí tu ventas_normalizado.csv

# Cargar dataset
df = pd.read_csv('ventas_normalizado.csv')
print(f"✅ Dataset cargado: {df.shape[0]} filas | {df.shape[1]} columnas")
df.head(3)

✅ API Key cargada desde Secrets


Saving ventas_normalizado.csv to ventas_normalizado (1).csv
✅ Dataset cargado: 2823 filas | 24 columnas


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,PHONE,ADDRESSLINE1,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2003-02-24,SHIPPED,1,2,2003,...,2125557818,897 Long Airport Avenue,NYC,NY,10022,USA,UNKNOWN,Yu,Kwai,SMALL
1,10121,34,81.35,5,2765.90,2003-05-07,SHIPPED,2,5,2003,...,26.47.1555,59 rue de l'Abbaye,Reims,UNKNOWN,51100,FRANCE,EMEA,Henriot,Paul,SMALL
2,10134,41,94.74,2,3884.34,2003-07-01,SHIPPED,3,7,2003,...,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,Paris,UNKNOWN,75508,FRANCE,EMEA,Da Cunha,Daniel,MEDIUM


Bloque 5 — Crear Documentos RAG

In [5]:
documentos = []

for _, fila in df.iterrows():
    contenido = f"""
    Orden: {fila['ORDERNUMBER']} | Fecha: {fila['ORDERDATE']}
    Cliente: {fila['CUSTOMERNAME']} | País: {fila['COUNTRY']}
    Producto: {fila['PRODUCTLINE']} | Cantidad: {fila['QUANTITYORDERED']}
    Precio unitario: {fila['PRICEEACH']} | Venta total: {fila['SALES']}
    Estado: {fila['STATUS']} | Tamaño deal: {fila['DEALSIZE']}
    """
    documentos.append(Document(page_content=contenido))

print(f"✅ {len(documentos)} documentos creados listos para RAG")

✅ 2823 documentos creados listos para RAG


Bloque 6 — Embeddings y Base Vectorial FAISS

In [6]:
embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    api_key=os.environ["MISTRAL_API_KEY"]
)

vectorstore = None
tamano_lote = 100

for i in range(0, len(documentos), tamano_lote):
    lote = documentos[i:i + tamano_lote]
    if vectorstore is None:
        vectorstore = FAISS.from_documents(lote, embeddings)
    else:
        vectorstore.add_documents(lote)
    print(f"⏳ Procesados {min(i + tamano_lote, len(documentos))}/{len(documentos)} documentos")
    time.sleep(1)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("\n✅ Base vectorial FAISS lista")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

⏳ Procesados 100/2823 documentos
⏳ Procesados 200/2823 documentos
⏳ Procesados 300/2823 documentos
⏳ Procesados 400/2823 documentos
⏳ Procesados 500/2823 documentos
⏳ Procesados 600/2823 documentos
⏳ Procesados 700/2823 documentos
⏳ Procesados 800/2823 documentos
⏳ Procesados 900/2823 documentos
⏳ Procesados 1000/2823 documentos
⏳ Procesados 1100/2823 documentos
⏳ Procesados 1200/2823 documentos
⏳ Procesados 1300/2823 documentos
⏳ Procesados 1400/2823 documentos
⏳ Procesados 1500/2823 documentos
⏳ Procesados 1600/2823 documentos
⏳ Procesados 1700/2823 documentos
⏳ Procesados 1800/2823 documentos
⏳ Procesados 1900/2823 documentos
⏳ Procesados 2000/2823 documentos
⏳ Procesados 2100/2823 documentos
⏳ Procesados 2200/2823 documentos
⏳ Procesados 2300/2823 documentos
⏳ Procesados 2400/2823 documentos
⏳ Procesados 2500/2823 documentos
⏳ Procesados 2600/2823 documentos
⏳ Procesados 2700/2823 documentos
⏳ Procesados 2800/2823 documentos
⏳ Procesados 2823/2823 documentos

✅ Base vectorial FAISS

Bloque 7 — Configurar Mistral LLM

In [7]:
llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=os.environ["MISTRAL_API_KEY"],
    temperature=0.3
)

print("✅ Mistral LLM configurado")
print("📌 Modelo: mistral-small-latest")

✅ Mistral LLM configurado
📌 Modelo: mistral-small-latest


Bloque 8 — Crear las Tools

In [8]:

@tool
def ventas_por_pais(pais: str) -> str:
    """Calcula el total de ventas para un país específico."""
    pais = pais.upper().strip()
    resultado = df[df['COUNTRY'] == pais]['SALES'].sum()
    if resultado == 0:
        return f"No encontré ventas para el país: {pais}"
    return f"Total de ventas para {pais}: ${resultado:,.2f}"

@tool
def top_clientes(n: int = 5) -> str:
    """Devuelve los N clientes con mayor volumen de ventas."""
    top = df.groupby('CUSTOMERNAME')['SALES'].sum().nlargest(n)
    resultado = "\n".join([f"{i+1}. {cliente}: ${ventas:,.2f}"
                          for i, (cliente, ventas) in enumerate(top.items())])
    return f"Top {n} clientes:\n{resultado}"

@tool
def ventas_por_producto(producto: str) -> str:
    """Calcula el total de ventas para una línea de producto."""
    producto = producto.upper().strip()
    resultado = df[df['PRODUCTLINE'] == producto]['SALES'].sum()
    if resultado == 0:
        return f"No encontré ventas para: {producto}"
    return f"Total de ventas para {producto}: ${resultado:,.2f}"

@tool
def ordenes_por_estado(estado: str) -> str:
    """Cuenta las órdenes según su estado: SHIPPED, CANCELLED, ON HOLD, etc."""
    estado = estado.upper().strip()
    resultado = df[df['STATUS'] == estado].shape[0]
    return f"Hay {resultado} órdenes con estado: {estado}"

@tool
def resumen_general(_: str = "") -> str:
    """Devuelve un resumen general del dataset de ventas."""
    total_ventas = df['SALES'].sum()
    total_ordenes = df['ORDERNUMBER'].nunique()
    total_clientes = df['CUSTOMERNAME'].nunique()
    total_paises = df['COUNTRY'].nunique()
    mejor_producto = df.groupby('PRODUCTLINE')['SALES'].sum().idxmax()
    return f"""
    📊 Resumen General:
    - Total ventas: ${total_ventas:,.2f}
    - Total órdenes únicas: {total_ordenes}
    - Total clientes: {total_clientes}
    - Total países: {total_paises}
    - Producto más vendido: {mejor_producto}
    """

tools = [ventas_por_pais, top_clientes, ventas_por_producto,
         ordenes_por_estado, resumen_general]

print("✅ Tools creadas:")
for t in tools:
    print(f"   🔧 {t.name}")

✅ Tools creadas:
   🔧 ventas_por_pais
   🔧 top_clientes
   🔧 ventas_por_producto
   🔧 ordenes_por_estado
   🔧 resumen_general


Bloque 9 — Crear el Agente

In [9]:
agente_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt="""
    Eres un asistente experto en análisis de datos de ventas.
    Respondé siempre en español, de forma clara y concisa.
    Usá SIEMPRE la herramienta más adecuada para responder.
    Si ninguna herramienta aplica, respondé con el contexto disponible.
    """
)

print("✅ Agente ReAct creado correctamente")
print("📌 Herramientas disponibles:", [t.name for t in tools])

✅ Agente ReAct creado correctamente
📌 Herramientas disponibles: ['ventas_por_pais', 'top_clientes', 'ventas_por_producto', 'ordenes_por_estado', 'resumen_general']


Bloque 10 — Función BOT

In [10]:
def formatear_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

historial = []

def bot_agente(pregunta):
    # Recuperar contexto RAG
    docs = retriever.invoke(pregunta)
    contexto = formatear_docs(docs)

    # Mensaje con contexto RAG incluido
    mensaje_completo = f"""
    Contexto del dataset de ventas:
    {contexto}

    Pregunta: {pregunta}
    """

    # Ejecutar agente
    resultado = agente_executor.invoke({
        "messages": historial + [HumanMessage(content=mensaje_completo)]
    })

    # Extraer respuesta
    respuesta = resultado["messages"][-1].content

    # Guardar en historial
    historial.append(HumanMessage(content=pregunta))
    historial.append(AIMessage(content=respuesta))

    return respuesta

print("✅ Función BOT lista")

✅ Función BOT lista


Bloque 11 — Interfaz Widget

In [11]:

historial = []

titulo = HTML("<h3 style='color:#1b4080'>🤖 BOT de Ventas — Agente + RAG + Mistral</h3><hr>")
chat_output = widgets.Output()
texto_input = widgets.Text(
    placeholder='Escribí tu pregunta aquí...',
    layout=widgets.Layout(width='75%')
)
boton_enviar = widgets.Button(
    description='Enviar',
    button_style='primary',
    layout=widgets.Layout(width='12%')
)
boton_limpiar = widgets.Button(
    description='Limpiar',
    button_style='warning',
    layout=widgets.Layout(width='10%')
)

def mostrar_mensaje(rol, texto):
    if rol == "usuario":
        color, icono, alineacion = "#2d6a4f", "🙋", "right"
    else:
        color, icono, alineacion = "#1b4080", "🤖", "left"
    with chat_output:
        display(HTML(f"""
        <div style='text-align:{alineacion}; margin:8px 0'>
            <span style='background-color:{color}; color:white;
                         padding:10px 15px; border-radius:15px;
                         display:inline-block; max-width:80%;
                         text-align:left; font-size:14px'>
                {icono} {texto}
            </span>
        </div>
        """))

def al_enviar(b):
    pregunta = texto_input.value.strip()
    if not pregunta:
        return
    texto_input.value = ""
    mostrar_mensaje("usuario", pregunta)
    with chat_output:
        display(HTML("<p style='color:gray;font-size:12px'>⏳ Pensando...</p>"))
    respuesta = bot_agente(pregunta)
    mostrar_mensaje("bot", respuesta)

def al_limpiar(b):
    historial.clear()
    chat_output.clear_output()
    with chat_output:
        display(HTML("<p style='color:gray'>🗑️ Chat limpiado.</p>"))

boton_enviar.on_click(al_enviar)
boton_limpiar.on_click(al_limpiar)
texto_input.on_submit(al_enviar)

display(titulo)
display(chat_output)
display(widgets.HBox([texto_input, boton_enviar, boton_limpiar]))

with chat_output:
    display(HTML("""
    <p style='color:gray'>👋 ¡Hola! Soy tu asistente de ventas.<br>
    Podés preguntarme sobre clientes, países, productos o pedirme un resumen.</p>
    """))

Output()